# 02 — Cross-sectional momentum

Rank the universe on 9-month return skipping the most recent month, hold the top
quartile, and stand down in the state momentum crashes in.

**Why skip a month.** One-month returns reverse on average, so a momentum signal
that includes the latest month is partly buying the very thing that is about to
mean-revert. Jegadeesh & Titman skip it for exactly this reason.

**Why the crash filter.** Momentum's characteristic failure is not gradual
underperformance but a crash: it loses most in the rebound *after* a drawdown,
when the beaten-down names it is not long rally hardest. The state that predicts
it is observable in real time — market below its own 200-day average *and*
realized volatility elevated — so the strategy stands down there.

Standalone: no `portfolio_agent` import.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# Simulation settings, shared by every notebook so the strategies are comparable.
#
# execution_lag=1 is the property that keeps this honest: a signal computed from
# day t's close is traded into day t+1's return. The engine refuses lag=0.
config = L.BacktestConfig(
    initial_capital=1_000_000.0,
    cost_bps=25.0,        # all-in round trip for Indian cash equities
    max_weight=0.10,
    rebalance_days=5,     # weekly; the main control over turnover
    max_gross=1.0,        # long-only, unlevered
    execution_lag=1,
)

benchmark = L.equal_weight_benchmark(close, config)
print("equal-weight buy & hold:",
      {k: round(v, 4) for k, v in benchmark.stats.items()
       if k in ("cagr", "sharpe", "max_drawdown")})

## Signal

In [ ]:
momentum_scores = L.momentum_scores(
    feature_panel,
    top_fraction=0.25,   # hold the top quartile
    crash_filter=True,
    vol_target=0.25,
)

held = (momentum_scores > 0).sum(axis=1)
print(f"names held: mean {held.mean():.1f} | flat on {(held == 0).mean():.1%} of sessions")

In [ ]:
# What the crash filter costs and what it saves. The unfiltered version is the
# same signal with the stand-down removed.
unfiltered = L.momentum_scores(feature_panel, top_fraction=0.25, crash_filter=False)

filtered_result = L.run_backtest(momentum_scores, close, config)
unfiltered_result = L.run_backtest(unfiltered, close, config)

display(L.compare_stats({
    "with crash filter": filtered_result,
    "without": unfiltered_result,
    "equal weight": benchmark,
}))

L.plot_equity({"with crash filter": filtered_result, "without": unfiltered_result},
              title="Momentum: effect of the crash filter")

## Backtest

Against equal-weight buy-and-hold of the same names — the honest comparison for a long-only stock picker. Beating cash is not the question.

In [ ]:
result = L.run_backtest(momentum_scores, close, config)

comparison = L.compare_stats({"momentum": result, "equal weight": benchmark})
display(comparison)

## Analysis

In [ ]:
L.plot_equity({"momentum": result}, title="momentum vs equal weight",
              benchmark=benchmark.returns)

In [ ]:
L.plot_return_profile(result, "momentum")

In [ ]:
L.plot_exposure(result, "momentum")

In [ ]:
L.plot_weight_heatmap(result, title="momentum: allocation over time")

## Sensitivity

How concentrated should the book be? A narrower slice holds stronger names and
more idiosyncratic risk; a wider one converges on the index.

In [ ]:
sweep = {}
for fraction in (0.15, 0.25, 0.35, 0.50):
    scores = L.momentum_scores(feature_panel, top_fraction=fraction)
    sweep[f"top {fraction:.0%}"] = L.run_backtest(scores, close, config)

display(L.compare_stats(sweep)[["sharpe", "cagr", "max_drawdown", "avg_positions"]])
L.plot_stats_table(L.compare_stats(sweep), title="Concentration sensitivity")

In [ ]:
# Rebalance frequency trades signal freshness against cost. A signal this slow
# should not need daily trading, and daily trading of it mostly buys costs.
sweep = {}
for days in (1, 5, 21, 63):
    variant = L.BacktestConfig(cost_bps=config.cost_bps, max_weight=config.max_weight,
                               rebalance_days=days)
    sweep[f"every {days}d"] = L.run_backtest(momentum_scores, close, variant)

display(L.compare_stats(sweep)[["sharpe", "cagr", "ann_turnover"]])

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.